<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Changing code safely →](02_Changing_Code_Safely.ipynb)

# 1 · Programs as data

**The problem.** Code generators usually paste strings together. It works until it doesn't: an expression pasted into
a template loses its meaning when it lands next to a stronger operator, a name collides with a keyword, an indent is
off by one. Here's the classic mistake, a generator that doubles an expression:

In [ ]:
def doubled(expression):
    return f"def f(x):\n    return {expression} * 2\n"


print(doubled("x + 1"))

`x + 1 * 2` isn't twice `x + 1`. The generator needed to know Python's precedence, and so does every template that
pastes code. mbse-programs works on programs as **trees** instead: built, read and changed as data, and turned into
source text only at the end, by a printer that knows the language's grammar.

This case study parses source into a tree, looks at it, prints it, and builds one by hand.

In [ ]:
from mbse.Programs.Framework import Errors
from mbse.Programs.Python import Python314, Syntax as P
from toolkit import outline

## Step 1: parsing source into a tree

A **standard**, here `Python314`, parses source text into a tree and prints trees back. `outline` (from the toolkit)
shows the tree one syntax node per line: its kind, and under it each child, labeled with the property that holds it.

In [ ]:
module = Python314.parse("def area(width, height):\n    return width * height\n")
print(outline(module))

The kinds are Python's own abstract syntax, as the `ast` module has it: `FunctionDef`, `Arguments`, `Arg`, `Return`,
`BinOp`, `Name`. If you know `ast`, you know this tree. Where it departs from `ast`, it keeps what `ast` drops: names
are `Identifier` syntax nodes, so one traversal finds every name, and literals keep their spelling.

## Step 2: syntax nodes are plain objects

Each syntax node is an ordinary object with its properties. **Attributes** hold values (`op='*'`, an identifier's
`spelling`), and **children** hold other syntax nodes:

In [ ]:
function = module.body[0]
print(function.name.spelling, [arg.arg.spelling for arg in function.args.args])
product = function.body[0].value
print(type(product).__name__, product.op, product.left.id.spelling, product.right.id.spelling)

## Step 3: printing

Printing writes one fixed layout: four spaces per level, PEP 8's blank lines, parentheses only where the grammar
needs them. The tree doesn't record whitespace, so source comes back tidied:

In [ ]:
messy = Python314.parse("def  area( width,height ) :\n  return (width*height)\nx=area(2,3);y=area(4,5)\n")
print(Python314.print(messy))

The parentheses around `width*height` were written in the source, so the tree keeps them (as a `Parenthesized`
node); everything else is the printer's layout.

## Step 4: building a tree

Trees are built with each language's fluent **builders**, `P.LANGUAGE.Builders.<Kind>()`, with one setter per property.
A child's setter takes a syntax node, or a function that builds one. Here's the doubling generator again, written as a
tree: it takes the expression as a tree too, and multiplies it by two.

In [ ]:
Py = P.LANGUAGE.Builders


def doubled(expression):
    body = Py.Return().value(lambda b: b.BinOp().left(expression).op("*").right(lambda b: b.Constant().spelling("2")))
    return Py.FunctionDef().name("f").args(lambda a: a.add_args(lambda b: b.arg("x"))).add_body(body).create()


x_plus_1 = Python314.parse("x + 1\n").body[0].value
print(Python314.print(doubled(x_plus_1)))

The printer added the parentheses, because a `BinOp` of `+` under a `*` needs them. The generator didn't have to know.
A property whose kind has one attribute takes the attribute's value directly: `.name("f")` makes the `Identifier`, and
`.arg("x")` the parameter's.

## Step 5: a tree must be complete

`validate` reports what's missing or wrong, by path. A tree that isn't valid doesn't print:

In [ ]:
half_built = Py.BinOp().op("+").right(lambda b: b.Name().id("y")).create()
print(P.LANGUAGE.validate(half_built))
try:
    Python314.print(half_built)
except Errors.PrintError as error:
    print(error)

## Why it works this way

- **Text only at the edges.** Parsing and printing are the only places text appears. Everything in between, reading,
  changing and generating code, works on trees, so precedence, quoting and layout are the printer's job, done once.
- **The language's own abstract syntax.** Python's tree follows `ast`; TypeScript's follows typescript-estree, and C++'s
  the standard's grammar (case study 5). A programmer who knows the language's tools knows its tree.
- **Plain objects.** Syntax nodes are ordinary mutable objects, so a program reads and edits them like any other
  data: the next case study changes code in place.

<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Changing code safely →](02_Changing_Code_Safely.ipynb)